# 05｜溺水偵測引擎

執行 YOLO 追蹤、時間窗判斷、影像標註、CSV 紀錄與警示通知。


In [ ]:
def processing_worker(source, is_stream, conf_threshold, warning_ratio,
                       warning_window_sec, min_ratio_of_window):
    cap = None
    csv_file = None
    csv_writer = None

    line_enabled = state.line_enabled
    line_token = state.line_token

    try:
        cap = cv2.VideoCapture(source)
        if not cap.isOpened():
            state.update_stats(f"無法開啟來源：{source}", False)
            return

        fps_video = cap.get(cv2.CAP_PROP_FPS) or 30
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        total_sec = total_frames / fps_video if fps_video and total_frames > 0 else 0

        warning_window_frames = max(1, int(round(warning_window_sec * fps_video)))
        min_samples_required = max(1, int(round(warning_window_frames * min_ratio_of_window)))

        track_history = defaultdict(lambda: deque(maxlen=warning_window_frames))
        active_warning_ids = set()
        warned_ids_ever = set()
        warning_trigger_total = 0
        target_counts = {c: 0 for c in TARGET_CLASSES}

        csv_file = open(LOG_PATH, "w", newline="", encoding="utf-8-sig")
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow([
            "系統時間", "影片時間", "幀編號", "Tracking_ID", "檢測類別",
            "信心度(%)", "邊界框_X1", "邊界框_Y1", "邊界框_X2", "邊界框_Y2"
        ])
        state.log_ready = False

        frame_idx = 0
        prev_time = time.time()
        smoothed_fps = 0.0

        while state.running:
            if state.paused:
                time.sleep(0.1)
                continue

            ret, frame = cap.read()
            if not ret:
                break
            frame_idx += 1

            now = time.time()
            instant_fps = 1.0 / max(now - prev_time, 1e-6)
            smoothed_fps = instant_fps if smoothed_fps == 0 else (0.9 * smoothed_fps + 0.1 * instant_fps)
            prev_time = now

            results = model.track(
                frame, conf=conf_threshold, device=device,
                persist=True, tracker=TRACKER_YAML, verbose=False
            )
            result = results[0]

            active_warning_ids.clear()
            newly_triggered_ids = []

            wall_clock_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
            video_time_str = fmt_time(frame_idx / fps_video) if (not is_stream and fps_video) else ""

            if result.boxes is not None:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    cls_name = model.names[cls_id]
                    conf_pct = float(box.conf[0]) * 100
                    x1, y1, x2, y2 = [float(v) for v in box.xyxy[0]]
                    tid = int(box.id[0]) if box.id is not None else -1

                    if cls_name in target_counts:
                        target_counts[cls_name] += 1

                    csv_writer.writerow([
                        wall_clock_now, video_time_str, frame_idx,
                        tid, cls_name, f"{conf_pct:.1f}",
                        f"{x1:.1f}", f"{y1:.1f}", f"{x2:.1f}", f"{y2:.1f}"
                    ])

                    if box.id is not None:
                        track_history[tid].append(cls_name)
                        dq = track_history[tid]
                        total = len(dq)
                        if total >= min_samples_required:
                            drowning_count = sum(1 for c in dq if c == "Drowning")
                            ratio = drowning_count / total
                            if ratio > warning_ratio:
                                active_warning_ids.add(tid)
                                if tid not in warned_ids_ever:
                                    warned_ids_ever.add(tid)
                                    warning_trigger_total += 1
                                    newly_triggered_ids.append(tid)

            if frame_idx % 30 == 0:
                csv_file.flush()

            annotated = result.plot()

            if result.boxes is not None:
                for box in result.boxes:
                    if box.id is not None and int(box.id[0]) in active_warning_ids:
                        x1, y1, x2, y2 = map(int, box.xyxy[0])
                        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 0, 255), 4)
                        cv2.putText(annotated, "WARNING", (x1, max(0, y1 - 10)),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            annotated = draw_dashboard(annotated, smoothed_fps, target_counts,
                                        len(active_warning_ids), warning_trigger_total)
            if len(active_warning_ids) > 0:
                annotated = draw_warning_banner(annotated, width)

            # ★ 純文字通知：只在首次觸發時發送
            if newly_triggered_ids and line_enabled and line_token:
                if not is_stream:
                    time_desc = f"影片時間 {fmt_time(frame_idx / fps_video)}"
                else:
                    time_desc = f"系統時間 {datetime.now().strftime('%H:%M:%S')}"

                msg = (
                    f"⚠️ 發現疑似溺水者\n"
                    f"{time_desc}\n"
                    f"幀編號: {frame_idx}\n"
                    f"Tracking ID: {', '.join(map(str, newly_triggered_ids))}"
                )
                send_line_text_async(line_token, msg)

            disp_frame = annotated
            if width > DISPLAY_MAX_WIDTH:
                scale = DISPLAY_MAX_WIDTH / width
                disp_frame = cv2.resize(annotated, (int(width * scale), int(height * scale)))

            ok, buffer = cv2.imencode('.jpg', disp_frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
            if ok:
                state.update_frame(buffer.tobytes())

            cur_sec = frame_idx / fps_video if fps_video else 0
            display_clock = datetime.now().strftime("%H:%M:%S")
            if not is_stream:
                time_line = f"影片時間: {fmt_time(cur_sec)} / {fmt_time(total_sec)} | 系統時間: {display_clock}"
            else:
                time_line = f"串流已運行: {fmt_time(cur_sec)} | 系統時間: {display_clock}"

            progress = f"{frame_idx}/{total_frames}" if not is_stream else f"{frame_idx} (即時串流)"
            stats = (
                f"{time_line}\n"
                f"處理進度: {progress} | FPS: {smoothed_fps:.1f}\n"
                f"{'-'*35}\n"
            )
            for name in TARGET_CLASSES:
                stats += f"{name}: {target_counts.get(name, 0)}\n"
            stats += f"{'-'*35}\n累計警示物件數: {warning_trigger_total}"

            state.update_stats(stats, len(active_warning_ids) > 0)

        final_text, _, _, _, _ = state.get_stats()
        if "===== 處理完成 =====" not in final_text and "無法開啟" not in final_text:
            state.update_stats(final_text + "\n\n===== 處理已結束 =====", False)

    finally:
        if cap is not None:
            cap.release()
        if csv_file is not None:
            csv_file.flush()
            csv_file.close()
            state.log_ready = True
        state.running = False
        state.paused = False
